# Prepare Clinical data

In [1]:
BINARY_PFS = True
LONG_RESP_THRESHOLD = 6
DROP_BOR = True
RNA_SEQ_ONLY = True

In [2]:
import pandas as pd
import numpy as np

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.',
    "doi:10.3390/cancers12082224": 'Blateau et al.',
    "doi:10.1200/PO.16.00054": 'Catalanotti et al.',
    'doi:10.1158/2159-8290.CD-13-0617': 'Van Allen at al.'
}

clinical = pd.read_csv("../dataset/original/clinical.csv")
clinical['patientID'] = clinical['patientID'].str.replace(r'^HL_', 'HL_Shi-', regex=True)
clinical['source'] = clinical['source'].replace(source_map)
if RNA_SEQ_ONLY == True:
    clinical = clinical[clinical['source'].isin(['Hugo et al.', 'Kwong et al.', 'Yan et al.'])]
clinical = clinical.drop(columns=['id', 'creation_datetime', 'original_patientID', 'OS_status', 'OS_month', 'CNA_data', 'SNV_data', 'GEX_data'])

if DROP_BOR == True:
    clinical = clinical.drop(columns=['BOR'])

print(clinical.shape)
clinical.head()

(196, 14)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_status,PFS_month,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source
164,YR_3053,male,26,IV,M1B,elevated,1.0,2.9,vemurafenib,V600E,NaN,no,no,Yan et al.
165,YR_3151,female,28,IV,M1C,normal,1.0,1.9,vemurafenib,V600E,NaN,no,no,Yan et al.
166,YR_3708,female,58,IV,M1A,normal,0.0,13.2,vemurafenib,V600E,NaN,no,no,Yan et al.
167,YR_3705,male,45,IV,M1A,normal,0.0,17.4,vemurafenib,V600E,NaN,no,no,Yan et al.
168,YR_1006,male,53,IV,M1C,elevated,1.0,1.4,vemurafenib + cobimetinib,V600E,NaN,no,no,Yan et al.


In [3]:
clinical['source'].unique()

array(['Yan et al.', 'Kwong et al.', 'Hugo et al.'], dtype=object)

## Add Target label

In [4]:
# Check unexpected PFS_status values
print(clinical['PFS_status'].value_counts(dropna=False))

PFS_status
1.0    161
0.0     35
Name: count, dtype: int64


In [5]:
clinical['PFS_status'] = pd.to_numeric(clinical['PFS_status'], errors='coerce')

def label_pfs(row):
    if BINARY_PFS == False:
        if row['PFS_month'] >= LONG_RESP_THRESHOLD:
            return 2
        elif row['PFS_month'] < 6 and row['PFS_status'] == 1:
            return 0
        elif row['PFS_month'] >= 6 and row['PFS_month'] < LONG_RESP_THRESHOLD and row['PFS_status'] == 1:
            return 1
        return np.nan
    
    else:
        if row['PFS_month'] >= LONG_RESP_THRESHOLD:
            return 1
        elif row['PFS_month'] < 6 and row['PFS_status'] == 1:
            return 0
        return np.nan

clinical['pfs_label'] = clinical.apply(label_pfs, axis=1)
clinical = clinical.drop(columns=['PFS_status']).dropna(subset=['pfs_label'])
clinical['pfs_label'] = clinical['pfs_label'].astype(int)

print(clinical.shape)
clinical.head()


(195, 14)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label
164,YR_3053,male,26,IV,M1B,elevated,2.9,vemurafenib,V600E,NaN,no,no,Yan et al.,0
165,YR_3151,female,28,IV,M1C,normal,1.9,vemurafenib,V600E,NaN,no,no,Yan et al.,0
166,YR_3708,female,58,IV,M1A,normal,13.2,vemurafenib,V600E,NaN,no,no,Yan et al.,1
167,YR_3705,male,45,IV,M1A,normal,17.4,vemurafenib,V600E,NaN,no,no,Yan et al.,1
168,YR_1006,male,53,IV,M1C,elevated,1.4,vemurafenib + cobimetinib,V600E,NaN,no,no,Yan et al.,0


In [6]:
clinical['pfs_label'].value_counts().sort_index()

pfs_label
0    110
1     85
Name: count, dtype: int64

## OHE 'drug' and 'BRAF_mut'

In [7]:
drug_series = clinical['drug'].fillna('').str.replace(r'\s*\+\s*', '+', regex=True).str.strip()
drug_dummies = drug_series.str.get_dummies(sep='+').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('drug', axis=1), drug_dummies], axis=1)

braf_series = clinical['BRAF_mut'].fillna('').str.replace(r'\s*;\s*', ';', regex=True).str.strip()
braf_mut_dummies = braf_series.str.get_dummies(sep=';').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('BRAF_mut', axis=1), braf_mut_dummies], axis=1)
clinical = clinical.drop(columns=['nan'], errors='ignore')

print(clinical.shape)
clinical.head()

(195, 19)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,cobimetinib,dabrafenib,trametinib,vemurafenib,MND,V600E,V600K
164,YR_3053,male,26,IV,M1B,elevated,2.9,NaN,no,no,Yan et al.,0,0,0,0,1,0,1,0
165,YR_3151,female,28,IV,M1C,normal,1.9,NaN,no,no,Yan et al.,0,0,0,0,1,0,1,0
166,YR_3708,female,58,IV,M1A,normal,13.2,NaN,no,no,Yan et al.,1,0,0,0,1,0,1,0
167,YR_3705,male,45,IV,M1A,normal,17.4,NaN,no,no,Yan et al.,1,0,0,0,1,0,1,0
168,YR_1006,male,53,IV,M1C,elevated,1.4,NaN,no,no,Yan et al.,0,1,0,0,1,0,1,0


In [8]:
clinical.to_csv(f"../dataset/created/clinical.csv", index=False)